# Epi Info AI advanced spatial K09 validation lab — V0.1

This notebook is an independent Python oracle for the K09 corrective follow-up. It re-derives Moran's I, conditional LISA permutations, and standard Getis-Ord Gi* from a small hand-audited chain. It does not import or call the TypeScript implementation. Passing is evidence, not validation or legacy parity.

In [ ]:
import math
values = [1.0, 2.0, 8.0]
ids = ['a', 'b', 'c']
weights = {'a': {'a': 1.0, 'b': 1.0}, 'b': {'b': 1.0, 'a': 0.5, 'c': 0.5}, 'c': {'c': 1.0, 'b': 1.0}}
mean = sum(values) / len(values)
variance = sum((value - mean) ** 2 for value in values) / len(values)
deviations = dict(zip(ids, (value - mean for value in values)))
moran_numerator = sum(weight * deviations[left] * deviations[right] for left, row in {'a': {'b': 1.0}, 'b': {'a': 0.5, 'c': 0.5}, 'c': {'b': 1.0}}.items() for right, weight in row.items())
moran = moran_numerator / sum(deviations[id] ** 2 for id in ids)
assert math.isclose(moran, -0.14534883720930236, rel_tol=0, abs_tol=1e-12)
assert math.isclose(-1 / (len(values) - 1), -0.5, rel_tol=0, abs_tol=1e-12)
{'moranI': moran, 'expectedRandomization': -1 / (len(values) - 1)}

In [ ]:
def gi_star(identifier):
    row = weights[identifier]
    weight_sum = sum(row.values())
    squared_sum = sum(weight * weight for weight in row.values())
    weighted_sum = sum(row[neighbor] * values[ids.index(neighbor)] for neighbor in row)
    numerator = weighted_sum - mean * weight_sum
    denominator = math.sqrt(variance) * math.sqrt((len(values) * squared_sum - weight_sum ** 2) / (len(values) - 1))
    return numerator / denominator
gi = {identifier: gi_star(identifier) for identifier in ids}
expected_gi = {'a': -1.4018260516446992, 'b': -0.5391638660171918, 'c': 0.8626621856275073}
for identifier in ids: assert math.isclose(gi[identifier], expected_gi[identifier], rel_tol=0, abs_tol=1e-12)
gi

In [ ]:
def lisa(identifier, permuted):
    focal = permuted[ids.index(identifier)] - mean
    lag = sum(weights[identifier][neighbor] * (permuted[ids.index(neighbor)] - mean) for neighbor in weights[identifier] if neighbor != identifier)
    return focal * lag / variance
# Conditional null: keep the focal observation fixed and permute only the other values.
assert lisa('a', [values[0], values[2], values[1]]) != lisa('a', [values[1], values[0], values[2]])
observed = {identifier: lisa(identifier, values) for identifier in ids}
assert math.isclose(observed['a'], 0.46511627906976727, rel_tol=0, abs_tol=1e-12)
assert math.isclose(observed['b'], -0.14534883720930233, rel_tol=0, abs_tol=1e-12)
assert math.isclose(observed['c'], -0.755813953488372, rel_tol=0, abs_tol=1e-12)
{'conditionalPermutation': 'focal value fixed', 'localI': observed}

In [ ]:
# Coordinates are interpreted as signed WGS84 longitude/latitude pairs.
def valid_coordinate(value):
    return isinstance(value, (list, tuple)) and len(value) == 2 and all(isinstance(item, (int, float)) and math.isfinite(item) for item in value) and -180 <= value[0] <= 180 and -90 <= value[1] <= 90
assert not valid_coordinate([181, 0])
assert not valid_coordinate([0, 91])
assert valid_coordinate([0, 90])
# Resource policy: permutation and cell-observation work must remain bounded.
assert 10_000 * 10_000 > 5_000_000
{'status': 'PASS', 'oracle': 'independent-python', 'typescript_imported': False}

## Interpretation

This lab is a self-correction oracle for formula and boundary regressions. It does not promote any K09 operation, establish Epi Info desktop parity, or close privacy, browser, Rust/WASM, or maintainer gates.